# Classificazione di malattie nelle foglie — PlantVillage

## Obiettivo

Riconoscere le malattie nelle foglie di pomodoro combinando un metodo non supervisionato e uno
supervisionato.

Il lavoro affronta **due fasi**:

1. **Autoencoder** (non supervisionato). Un autoencoder viene addestrato a ricostruire
   *solo foglie sane*

2. **Classificazione XGboost** (supervisionato). Tra 10 condizioni possibili (9 malattie + sana), un classificatore XGBoost assegna il tipo a partire dalle rappresentazioni apprese dall'encoder.

# Configurazione

In [ ]:
import os
os.makedirs("ckpt", exist_ok=True)
os.environ["KERAS_BACKEND"] = "torch"

In [ ]:
import keras
from keras import Sequential, Input
from keras.layers import Conv2D, MaxPooling2D, UpSampling2D, Rescaling
from keras.optimizers import Adam
from keras.losses import MeanSquaredError
from keras.metrics import MeanAbsoluteError
from keras.activations import leaky_relu
from keras.callbacks import EarlyStopping, ModelCheckpoint

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, glob, random, zipfile
from PIL import Image
from scipy.ndimage import gaussian_filter

import optuna
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from xgboost import XGBClassifier

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
keras.backend.backend()

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)

In [ ]:
KAGGLE_SLUG   = "abdallahalidev/plantvillage-dataset"
BASE_DIR      = "plantvillage"
TARGET_PLANT  = "Tomato"
USE_SEGMENTED = True  
IMG_SIZE      = 128   
BOTTLENECK_CH = 32    
TOPK_FRAC     = 0.02  # frazione di pixel peggiori usata come punteggio di anomalia

# Caricamento del dataset

Le immagini sono organizzate in cartelle nominate `Pianta___Condizione`, dove la condizione è `healthy`
per le foglie sane oppure il nome della malattia. Si usano le immagini a colori. La specie e la condizione
si ricavano dal nome della cartella.

In [ ]:
if not (os.path.isdir(BASE_DIR) and os.listdir(BASE_DIR)):
    # Percorso automatico: cerca lo zip del dataset in piu' posizioni e con piu' nomi,
    # cosi' il notebook gira senza modificare i path a mano.
    patterns = ["archive*.zip", "plantvillage*.zip", "*plantvillage*.zip", "plantvillage-dataset*.zip"]
    search_dirs = [".", "..", os.path.join("..", ".."),
                   os.path.expanduser("~/Downloads"), os.path.expanduser("~")]
    zip_path = None
    for _d in search_dirs:
        hits = []
        for _pat in patterns:
            hits += glob.glob(os.path.join(_d, _pat))
        if hits:
            zip_path = sorted(hits)[0]
            break
    if zip_path is None:
        raise FileNotFoundError(
            "Zip del dataset non trovato. Scaricalo da Kaggle "
            "(abdallahalidev/plantvillage-dataset) e lascialo in questa cartella "
            "o in ~/Downloads. Vedi README."
        )
    print("Estrazione dataset da:", zip_path)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(BASE_DIR)
    print("Dataset estratto in:", BASE_DIR)
else:
    print("Dataset gia' presente in:", BASE_DIR)

In [ ]:
target_folder = "segmented" if USE_SEGMENTED else "color"
class_dirs = [d for d in glob.glob(os.path.join(BASE_DIR, "**", "*___*"), recursive=True)
              if os.path.isdir(d)]
parents = {os.path.dirname(d) for d in class_dirs}
chosen = [p for p in parents if os.path.basename(p).lower() == target_folder]
DATA_DIR = chosen[0] if chosen else sorted(parents)[0]
DATA_DIR

In [ ]:
def parse_class(folder_name):
    """Dal nome cartella 'Pianta___Condizione' ricava (pianta, condizione, sana)."""
    plant, _, condition = folder_name.partition("___")
    plant = plant.split("_(")[0].replace("_", " ").strip()
    healthy = condition.lower() == "healthy"
    return plant, condition, healthy

In [ ]:
records = []
for folder in sorted(os.listdir(DATA_DIR)):
    folder_path = os.path.join(DATA_DIR, folder)
    if not os.path.isdir(folder_path) or "___" not in folder:
        continue
    plant, condition, healthy = parse_class(folder)
    for filename in os.listdir(folder_path):
        if filename.lower().endswith((".jpg", ".jpeg", ".png")):
            records.append({"path": os.path.join(folder_path, filename),
                            "plant": plant, "condition": condition, "healthy": healthy})

df = pd.DataFrame(records)

In [ ]:
print("Immagini totali:", len(df))
print("Specie:", df["plant"].nunique())
print("Classi:", df["condition"].nunique())
df.head()

# Panoramica del dataset

Numerosità per specie, proporzione sane/malate e distribuzione delle condizioni del pomodoro, la specie con
più malattie distinte e quindi la più adatta per entrambe le analisi.

In [ ]:
df["plant"].value_counts().sort_values().plot.barh(figsize=(8, 6), title="Immagini per specie")
plt.xlabel("numero di immagini")
plt.show()

In [ ]:
counts = df["healthy"].map({True: "Sane", False: "Malate"}).value_counts()
counts.plot.bar(title="Foglie sane vs malate", color=["#d62728", "#2ca02c"])
plt.ylabel("numero di immagini")
plt.show()

In [ ]:
df[df["plant"] == TARGET_PLANT]["condition"].value_counts().plot.barh(
    figsize=(8, 5), title=f"Condizioni per la specie: {TARGET_PLANT}")
plt.xlabel("numero di immagini")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
examples = pd.concat([
    df[(df.plant == TARGET_PLANT) & (df.healthy)].sample(4, random_state=SEED),
    df[(df.plant == TARGET_PLANT) & (~df.healthy)].sample(4, random_state=SEED),
])
for ax, (_, row) in zip(axes.ravel(), examples.iterrows()):
    ax.imshow(Image.open(row["path"]).convert("RGB"))
    ax.set_title("sana" if row["healthy"] else row["condition"], fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

# Preparazione delle immagini

Le immagini vengono lette, ridimensionate a `IMG_SIZE` e mantenute come pixel grezzi in **[0, 255]** --> la
normalizzazione è affidata a un layer `Rescaling` interno all'autoencoder. Ogni immagine ha forma
`(IMG_SIZE, IMG_SIZE, 3)`, con i canali in ultima posizione come richiesto dai layer convoluzionali di Keras.

In [ ]:
def load_images(frame, size=IMG_SIZE):
    """Carica le immagini di un DataFrame come array (N, size, size, 3) in [0, 255]."""
    out = np.empty((len(frame), size, size, 3), dtype="float32")
    for i, path in enumerate(frame["path"].values):
        img = Image.open(path).convert("RGB").resize((size, size))
        out[i] = np.asarray(img, dtype="float32")
    return out

# Autoencoder

L'autoencoder viene addestrato **solo sulle foglie sane** di pomodoro. Impara a comprimere e ricostruire
l'aspetto di una foglia sana. input e target coincidono.

In [ ]:
healthy_df = df[(df["plant"] == TARGET_PLANT) & (df["healthy"])]
train_df, val_df = train_test_split(healthy_df, test_size=0.2, random_state=SEED)
print("Foglie sane di addestramento:", len(train_df), "| validazione:", len(val_df))

In [ ]:
x_train = load_images(train_df)
x_val   = load_images(val_df)
x_train.shape, x_val.shape

Aumento dei dati con riflessioni orizzontali e verticali: triplica gli esempi sani senza introdurre
artefatti, rendendo la ricostruzione più stabile.

In [ ]:
x_train_aug = np.concatenate([x_train, x_train[:, :, ::-1, :], x_train[:, ::-1, :, :]], axis=0)
x_train_aug.shape

## Architettura

- **Encoder**: blocchi `Conv2D` + `MaxPooling2D`, risoluzione **128 → 64 → 32 → 16 → 8**, infine una
  convoluzione che riduce i canali al bottleneck.
- **Decoder**: simmetrico, `UpSampling2D` + `Conv2D`, risoluzione **8 → 16 → 32 → 64 → 128**; l'ultimo layer
  produce 3 canali con `leaky_relu` e un `Rescaling(255)` riporta i pixel in [0, 255]. La `leaky_relu`
  gestisce lo sfondo scuro delle immagini segmentate e, mantenendo sempre un piccolo gradiente.

In [ ]:
encoder = Sequential(
    [
        Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
        Rescaling(1 / 255),
        Conv2D(32, 3, padding="same", activation=leaky_relu),  MaxPooling2D(),
        Conv2D(64, 3, padding="same", activation=leaky_relu),  MaxPooling2D(),
        Conv2D(128, 3, padding="same", activation=leaky_relu), MaxPooling2D(),
        Conv2D(128, 3, padding="same", activation=leaky_relu), MaxPooling2D(),
        Conv2D(BOTTLENECK_CH, 3, padding="same", activation=leaky_relu),
    ],
    name="encoder",
)

In [ ]:
decoder = Sequential(
    [
        Input(shape=(8, 8, BOTTLENECK_CH)),
        Conv2D(128, 3, padding="same", activation=leaky_relu), UpSampling2D(),
        Conv2D(128, 3, padding="same", activation=leaky_relu), UpSampling2D(),
        Conv2D(64, 3, padding="same", activation=leaky_relu),  UpSampling2D(),
        Conv2D(32, 3, padding="same", activation=leaky_relu),  UpSampling2D(),
        Conv2D(3, 3, padding="same", activation=leaky_relu),
        Rescaling(255.0),
    ],
    name="decoder",
)

In [ ]:
autoencoder = Sequential([encoder, decoder])
autoencoder.summary()

## Addestramento

Loss `MeanSquaredError` tra immagine originale e ricostruita; metrica `MeanAbsoluteError`. `EarlyStopping`
interrompe quando la loss di validazione smette di migliorare e ripristina i pesi migliori.

In [ ]:
autoencoder.compile(
    optimizer=Adam(learning_rate=5e-4, clipnorm=1.0), 
    loss=MeanSquaredError(), 
    metrics=[MeanAbsoluteError()]
    )

In [ ]:
callbacks = [
    EarlyStopping(patience=12, restore_best_weights=True, monitor="val_loss"),  # tuned: 8 -> 12
    ModelCheckpoint("ckpt/autoencoder.weights.h5", save_weights_only=True, save_best_only=True, monitor="val_loss"),
]

In [ ]:
history = autoencoder.fit(
    x=x_train_aug, y=x_train_aug,
    epochs=100, batch_size=64, shuffle=True,  # tuned: batch_size 32 -> 64
    validation_data=(x_val, x_val),
    callbacks=callbacks,
)

In [ ]:
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="validation")
plt.xlabel("epoca"); plt.ylabel("MSE"); plt.title("Andamento della loss"); plt.legend()
plt.show()

## Ricostruzione delle foglie sane

Confronto tra foglie sane originali e la loro ricostruzione. Grazie al bottleneck convoluzionale, forma,
colore e venature principali sono riconoscibili e i contorni più definiti.

In [ ]:
def show_reconstructions(model, images, n=5):
    idxs = random.sample(range(len(images)), n)
    sample = images[idxs]
    recon = model.predict(sample, verbose=0)
    fig, axes = plt.subplots(n, 2, figsize=(5, 2.5 * n))
    for row in range(n):
        axes[row, 0].imshow((sample[row] / 255).clip(0, 1))
        axes[row, 1].imshow((recon[row] / 255).clip(0, 1))
        axes[row, 0].axis("off"); axes[row, 1].axis("off")
    axes[0, 0].set_title("originale"); axes[0, 1].set_title("ricostruita")
    plt.tight_layout(); plt.show()

show_reconstructions(autoencoder, x_val, n=5)

# Classificazione (XGBoost)

**Le caratteristiche.** Ogni foglia passa nell'encoder già addestrato, che la riassume nel bottleneck
`8 x 8 x BOTTLENECK_CH`; appiattito, diventa la riga di numeri su cui XGBoost impara a distinguere le malattie.

In [ ]:
parts = []
for condition, group in df[df["plant"] == TARGET_PLANT].groupby("condition"):
    parts.append(group.sample(n=min(500, len(group)), random_state=SEED))  # tuned: cap 250 -> 500
sup_df = pd.concat(parts).reset_index(drop=True)
sup_df["condition"].value_counts()

In [ ]:
x_sup = load_images(sup_df)
features = encoder.predict(x_sup, verbose=0)
X = features.reshape(len(features), -1)   # appiattisce il bottleneck spaziale
X.shape

In [ ]:
import gc, torch
del x_sup
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(sup_df["condition"].values)
list(label_encoder.classes_)

Suddivisione stratificata in train (60%), validation (20%) e test (20%)

In [ ]:
X_work, X_test, y_work, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_work, y_work, test_size=0.25, random_state=SEED, stratify=y_work)
print("train:", len(X_train), "| validation:", len(X_val), "| test:", len(X_test))

## Iperparametri con Optuna

Si cerca la combinazione di iperparametri che massimizza l'accuratezza sul validation, con ottimizzazione
bayesiana (Optuna). L'early stopping limita il numero di alberi.

In [ ]:
def objective(trial):
    params = {
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
    }
    model = XGBClassifier(
        n_estimators=600, eval_metric="mlogloss", early_stopping_rounds=20,
        tree_method="hist", device="cpu", n_jobs=-1, random_state=SEED, **params,
    )
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    return accuracy_score(y_val, model.predict(X_val))

In [ ]:
# tuned: sampler seminato (confronto riproducibile) + n_trials 20 -> 45
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=45, show_progress_bar=True)
study.best_params

In [ ]:
model = XGBClassifier(
    n_estimators=600, eval_metric="mlogloss", early_stopping_rounds=20,
    tree_method="hist", device="cpu", n_jobs=-1, random_state=SEED, **study.best_params,
)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

## Valutazione

L'**accuratezza** è la frazione di foglie classificate correttamente sul test set.

In [ ]:
y_pred = model.predict(X_test)
print("Accuratezza sul test:", round(accuracy_score(y_test, y_pred), 3))

La **matrice di confusione** mostra, per ogni malattia reale (riga), come sono state classificate le sue
foglie (colonna). La diagonale contiene le predizioni corrette; le celle fuori diagonale sono gli errori e
rivelano *quali* malattie si confondono tra loro.

In [ ]:
_, ax = plt.subplots(1, 1, figsize=(10, 9))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=label_encoder.classes_,
    cmap="Blues", ax=ax, xticks_rotation=90,
)
ax.set_title("Matrice di confusione — tipo di malattia")
plt.tight_layout(); plt.show()

In [ ]:
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))